<a href="https://colab.research.google.com/github/SimoRinaldi/Machine-Learning-Uni/blob/main/Data_analysis_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Analysis and Preprocessing

## Imports

The autoreload instruction reloads modules automatically before code execution, which is helpful for the update below.

In [ ]:
%load_ext autoreload
%autoreload 2

Make sure that we have the latest version of pandas-profiling and scikit-learn.

In [ ]:
import sys

!{sys.executable} -m pip install -U pandas-profiling[notebook]
!jupyter nbextension enable --py widgetsnbextension

#!pip uninstall scikit-learn -y
#!pip install -U scikit-learn
!pip install https://github.com/ydataai/pandas-profiling/archive/master.zip

In [ ]:
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn as sk
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.tree import DecisionTreeClassifier

import requests
import pandas_profiling as pp
from pandas_profiling.utils.cache import cache_file

In [ ]:
print(sk.__version__)

print(pd.__version__)

## Constants

In [ ]:
DATA_PATH = "https://raw.githubusercontent.com/serivan/mldmlab/master/Kaggle/KAGGLE/"
RANDOM_STATE = 3993
COLLINEAR_THRESHOLD = 0.9
K_FEATURES = 3

## Load Data

In [ ]:
dataset = pd.read_csv(DATA_PATH + "train.csv")

# Save the target column and drop the target
target = dataset["target"]
dataset.drop(columns="target", inplace=True)
# Sort columns by name
dataset.sort_index(axis="columns", inplace=True)
# Print first samples
dataset

## Missing Values

Reasons for missing values
- Information is not collected (e.g., people decline to give their age and weight)
- Attributes may not be applicable to all cases (e.g., annual income is not applicable to children)

Handling missing values
- Eliminate Data Objects
- Estimate Missing Values
- Ignore the Missing Value During Analysis
- Replace with all possible values (weighted by their probabilities)

In [ ]:
def count_missing(ds: pd.DataFrame):
    return ds.shape[0] - ds.count()

In [ ]:
count_missing(dataset)

In [ ]:
# Better visualization for missing count sorted by column
count_missing(dataset).sort_values().plot.barh()

In [ ]:
def get_sample_with_nans(ds: pd.DataFrame):
    return ds.loc[ds.isna().any(axis=1)]

In [ ]:
get_sample_with_nans(dataset)

In [ ]:
sample_with_nans_ids = get_sample_with_nans(dataset).index
sample_with_nans_ids

### Eliminate Data Objects

In [ ]:
dataset_no_nans = dataset.dropna()
dataset_no_nans

In [ ]:
count_missing(dataset_no_nans)

In [ ]:
try:
    dataset_no_nans.loc[sample_with_nans_ids]
except KeyError:
    print("Ids of desired samples are not in dataset!")

### Impute Missing Values

Estimate missing values can be performed using both pandas and scikit-learn.
We use scikit-learn because it helps with machine learning.

In [ ]:
def array_to_dataframe(arr: np.ndarray, columns: List[str]):
    return pd.DataFrame(data=arr, columns=columns)

#### Impute with constant

In [ ]:
constant_imp_arr = SimpleImputer(strategy="constant", fill_value=-1).fit_transform(
    dataset
)
constant_imp_arr

In [ ]:
constant_imp_df = array_to_dataframe(constant_imp_arr, dataset.columns)

In [ ]:
count_missing(constant_imp_df)

In [ ]:
constant_imp_df.loc[sample_with_nans_ids]

#### Impute with mean or median

In [ ]:
median_imp_arr = SimpleImputer(strategy="median").fit_transform(dataset)
median_imp_df = array_to_dataframe(arr=median_imp_arr, columns=dataset.columns)
median_imp_df

In [ ]:
count_missing(median_imp_df)

In [ ]:
median_imp_df.loc[sample_with_nans_ids]

#### Impute with KNearestNeighbours

In [ ]:
knn_imp_arr = KNNImputer().fit_transform(dataset)
knn_imp_df = array_to_dataframe(arr=knn_imp_arr, columns=dataset.columns)
knn_imp_df

In [ ]:
count_missing(knn_imp_df)

In [ ]:
knn_imp_df.loc[sample_with_nans_ids]

#### Impute with IterativeImputer ([BayesianRidge](https://www.tutorialspoint.com/scikit_learn/scikit_learn_bayesian_ridge_regression.htm))

In [ ]:
iter_imp_arr = IterativeImputer().fit_transform(dataset)
iter_imp_df = array_to_dataframe(arr=iter_imp_arr, columns=dataset.columns)
iter_imp_df

In [ ]:
count_missing(iter_imp_df)

In [ ]:
iter_imp_df.loc[sample_with_nans_ids]

## Aggregation

Combining two or more attributes (or objects) into a single attribute (or object)

Purpose
- Data reduction: Reduce the number of attributes or objects
- Change of scale: Cities aggregated into regions, states, countries, etc
- More “stable” data: Aggregated data tends to have less variability



See also https://www.tutorialspoint.com/scikit_learn/scikit_learn_bayesian_ridge_regression.htm

In [ ]:
dataset.groupby([ 'sex', 'chest_pain_type']).size().unstack()

Another more flexible way is to use crosstab():

In [ ]:
pd.crosstab(dataset['sex'], dataset['chest_pain_type'])


In [ ]:
pd.crosstab(dataset['sex'], dataset['chest_pain_type'], margins=True)

In [ ]:
pd.crosstab(dataset['sex'], dataset['chest_pain_type'], margins=True, normalize=True)

You can use groupby() with describe() for group summary statistics

In [ ]:
dataset.groupby('sex')['chest_pain_type'].describe()

In [ ]:
dataset.groupby('sex')['chest_pain_type'].describe()

You can use agg()/aggregate() for flexible aggregations

In [ ]:
dataset.groupby(['sex']).agg({'chest_pain_type': ['mean', 'std']})

In [ ]:
dataset.groupby(['sex']).agg({'chest_pain_type': ['mean', 'std'], 'cholesterol': ['mean', 'std']})

and take advantage of pivot_table()

In [ ]:
dataset.pivot_table(values='cholesterol', index='sex', columns='chest_pain_type')

###  Data reduction

#### Drop collinear columns

In [ ]:
def get_collinear_cols(ds: pd.DataFrame, coll_threshold: float):
    # Compute correlation matrix using pearson method (linear correlation)
    corr = iter_imp_df.corr(method="pearson")
    # Find collinear columns
    coll_cols = corr[corr > coll_threshold].dropna(thresh=2).dropna(axis="columns")
    return coll_cols

In [ ]:
get_collinear_cols(ds=dataset, coll_threshold=COLLINEAR_THRESHOLD)

In [ ]:
# Drop collinear column
iter_imp_df.drop(columns=["pulse"], inplace=True)
iter_imp_df

## Types of Sampling

- *Simple Random Sampling*: There is an equal probability of selecting any particular item
    - Sampling without replacement: As each item is selected, it is removed from the population
    - Sampling with replacement: Objects are not removed from the population as they are selected for the sample.In sampling with replacement, the same object can be picked up more than once
- *Stratified sampling*: Split the data into several partitions; then draw random samples from each partition

### Random Sampling

Random samplig 20% of the data without replacement

In [ ]:
def get_repeated_index(ds: pd.DataFrame):
    return ds.index.value_counts()[ds.index.value_counts() > 1]

In [ ]:
randsamp = iter_imp_df.sample(frac=0.2, random_state=RANDOM_STATE)
randsamp

In [ ]:
get_repeated_index(randsamp)

Random sampling 20% of the data with replacement

In [ ]:
randsamp_with_rep = iter_imp_df.sample(
    frac=0.2, replace=True, random_state=RANDOM_STATE
)
randsamp_with_rep

In [ ]:
get_repeated_index(randsamp_with_rep)

### Stratified Sampling

In [ ]:
def plot_pie_values_col(ds: pd.DataFrame, column: str):
    ds[column].value_counts()[ds[column].value_counts() > 3].plot.pie(
        figsize=(8, 6), autopct="%1.1f%%"
    )
    plt.show()

In [ ]:
plot_pie_values_col(ds=iter_imp_df, column="sex")

In [ ]:
plot_pie_values_col(ds=randsamp, column="sex")

In [ ]:
stratsamp = iter_imp_df.groupby("sex", group_keys=False).apply(
    lambda x: x.sample(frac=0.2)
)
stratsamp

In [ ]:
plot_pie_values_col(stratsamp, column="sex")

## Principal Component Analysis

In [ ]:
# Decomposition of PCA
pca_decomp = PCA(n_components=2, whiten=True, random_state=RANDOM_STATE)
pca_decomp = pca_decomp.fit(iter_imp_df)

# Dataset PCA as Numpy array
iter_imp_pca_arr = pca_decomp.transform(iter_imp_df)

In [ ]:
# Plot the pca clustered by age
sns.scatterplot(
    x=iter_imp_pca_arr[:, 0], y=iter_imp_pca_arr[:, 1], hue=iter_imp_df["age"]
)

## Feature Subset Selection

Techniques:
- Brute-force approach: Try all possible feature subsets as input to data mining algorithm
- Embedded approaches: Feature selection occurs naturally as part of the data mining algorithm
- Filter approaches: Features are selected before data mining algorithm is run
- Wrapper approaches: Use the data mining algorithm as a black box to find best subset of attributes

Techniques:
- Univariate approach:
    - KBest: Select features according to the k highest scores in univariate fashion (default using Analysis of Variance, aka ANOVA, scores).
- ModelBased approach: Feature selection using an estimator for feature ranking
    - Recursive Feature Elimination: Given an external estimator that assigns weights to features (e.g., the coefficients of a linear model), the goal of recursive feature elimination (RFE) is to select features by recursively considering smaller and smaller sets of features.
    - Sequential Feature Selection: This Sequential Feature Selector adds (forward selection) or removes (backward selection) features to form a feature subset in a greedy fashion.
    - Select From Model: Meta-transformer for selecting features based on importance weights.

### Univariate Approach

In [ ]:
def get_feature_univ_scores(univ_fselector, columns, sort_by: str = "score"):
    return pd.DataFrame(
        data=[univ_fselector.scores_, univ_fselector.pvalues_],
        index=["score", "pvalue"],
        columns=columns,
    ).T.sort_values(by=sort_by, ascending=False)

#### Select KBest

In [ ]:
# Create the feature selector
kbest = SelectKBest(k=3)
kbest.fit(iter_imp_df, target)

In [ ]:
# Print selected columns
iter_imp_df.columns[kbest.get_support()]

In [ ]:
# Feature selection scores and pvalues analysis
get_feature_univ_scores(kbest, iter_imp_df.columns, sort_by="score")

In [ ]:
# Transform dataset selection columns
kbest.transform(iter_imp_df)

### ModelBased Approach

#### Recursive Feature Elimination (Based On Regression)

In [ ]:
def get_feature_rfe_ranking(
    mb_fselector, columns,
):
    return pd.DataFrame(
        data=[mb_fselector.ranking_], index=["ranking"], columns=columns,
    ).T.sort_values(by="ranking", ascending=True)

In [ ]:
# Create the RecursiveFeatureElimination selector
rfe = RFE(
    estimator=LinearRegression(), n_features_to_select=K_FEATURES, step=1)
rfe.fit(iter_imp_df, target)

In [ ]:
# Print selected columns
iter_imp_df.columns[rfe.support_]

In [ ]:
# Analyze feature ranking
get_feature_rfe_ranking(rfe, iter_imp_df.columns)

In [ ]:
rfe.transform(iter_imp_df)

#### Sequential Feature Selection (Based on Classification)

In [ ]:
# Create the RecursiveFeatureElimination selector
sfs = SequentialFeatureSelector(
    estimator=DecisionTreeClassifier(max_depth=4),
    n_features_to_select=K_FEATURES,
    cv=5,
)
sfs.fit(iter_imp_df, target)

In [ ]:
# Print selected columns
iter_imp_df.columns[sfs.support_]

In [ ]:
sfs.transform(iter_imp_df)

#### Select From Model

In [ ]:
# Create the SelectFromModel selector
sfm = SelectFromModel(estimator=Lasso(), max_features=K_FEATURES)
sfm.fit(iter_imp_df, target)

In [ ]:
# Print selected columns
iter_imp_df.columns[sfm.get_support()]

In [ ]:
sfm.transform(iter_imp_df)

## Feature Construction

In [ ]:
# Create a new feature called relative_bp_s as the ratio between resting blood pressure and cholesterol
iter_imp_df["bp_s_chol"] = iter_imp_df["resting_bp_s"] / (
    iter_imp_df["cholesterol"] + 1e-7
)
iter_imp_df["bp_s_chol"]

## Attribute Transformation

In [ ]:
# Apply logaritmic transformation and create a new transformed column
iter_imp_df["transformed_mhr"] = iter_imp_df["max_heart_rate"].apply(
    lambda x: max(min(x - x / np.sqrt(x), 120), 90)
)

In [ ]:
# Visualize the current transformation
iter_imp_df[["max_heart_rate", "transformed_mhr"]].sort_values(
    by="max_heart_rate", ignore_index=True
).plot.line()

## Data profiling


In [ ]:
!pip freeze |grep pandas-profiling


In [ ]:
pp.ProfileReport(dataset)

In [ ]:
pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

In [ ]:
# Generate Pandas Profiling Report
profile = ProfileReport(dataset, explorative=True)
display(profile)